# AC-MOT — Run All v2

SCI-only Optuna tuning. Fixed YOLOv8n + fixed ByteTrack. Validation only for tuning; frozen weights before final test.

In [ ]:

from google.colab import drive, auth
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

drive.mount('/content/drive', force_remount=False)

DATA_ROOT = Path('/content/drive/MyDrive/AC-MOT-data')
RESULT_ROOT = Path('/content/drive/MyDrive/AC-MOT-results/optuna_sci_only')
VAL_DIR = DATA_ROOT / 'VisDrone2019-MOT-val'
TEST_SHARED_FOLDER_ID = '13hmlOtSDCsCt_3ORo_MhhujdMWiC2Qm2'
TEST_TARGET = Path(f'/content/drive/.shortcut-targets-by-id/{TEST_SHARED_FOLDER_ID}')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

def valid(p):
    p=Path(p)
    return p.is_dir() and (p/'sequences').is_dir() and (p/'annotations').is_dir()

if not valid(VAL_DIR):
    url='https://huggingface.co/datasets/vanthanh/VisDrone2019-MOT/resolve/main/VisDrone2019-MOT-val.zip'
    z=DATA_ROOT/'VisDrone2019-MOT-val.zip'
    subprocess.run(['wget','-c','-O',str(z),url],check=True)
    with zipfile.ZipFile(z,'r') as f: f.extractall(DATA_ROOT)
    z.unlink(missing_ok=True)
if not valid(VAL_DIR): raise RuntimeError(f'Validation not found: {VAL_DIR}')

if not valid(TEST_TARGET):
    auth.authenticate_user()
    from google.auth import default
    from googleapiclient.discovery import build
    creds,_=default(); svc=build('drive','v3',credentials=creds,cache_discovery=False)
    q=("name='VisDrone2019-MOT-test-dev' and mimeType='application/vnd.google-apps.shortcut' and trashed=false")
    items=svc.files().list(q=q,fields='files(id,shortcutDetails)').execute().get('files',[])
    if not any(x.get('shortcutDetails',{}).get('targetId')==TEST_SHARED_FOLDER_ID for x in items):
        svc.files().create(body={'name':'VisDrone2019-MOT-test-dev','mimeType':'application/vnd.google-apps.shortcut','shortcutDetails':{'targetId':TEST_SHARED_FOLDER_ID},'parents':['root']},fields='id').execute()
    time.sleep(4)
if not valid(TEST_TARGET):
    raise RuntimeError('Shared Test-dev still not visible. Add it as a shortcut to My Drive once, then rerun.')

print('[OK] Validation:', VAL_DIR)
print('[OK] Test-dev:', TEST_TARGET)
print('[OK] Validation sequences:', len([p for p in (VAL_DIR/'sequences').iterdir() if p.is_dir()]))
print('[OK] Test sequences:', len([p for p in (TEST_TARGET/'sequences').iterdir() if p.is_dir()]))


In [ ]:

# Fresh clone, then patch only the dataset plumbing in the existing SCI-only runner.
REPO=Path('/content/AC-MOT')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/AhmedCode110/AC-MOT.git',str(REPO)],check=True)
script=REPO/'scripts/optuna_sci_only_colab.py'
s=script.read_text()
s=s.replace("TEST_DIR = DATA_ROOT / 'VisDrone2019-MOT-test-dev'", f"TEST_DIR = Path('{TEST_TARGET}')")
s=s.replace("ensure_dataset(TEST_DIR, TEST_GDRIVE_ID)", "assert valid_dataset(TEST_DIR), f'Test-dev not found: {TEST_DIR}'")
old = "if TEST_LOCK.exists():\n    raise RuntimeError(f'Final test already exists: {TEST_LOCK}. Do not repeatedly retune against test.')"
new = "if TEST_LOCK.exists():\n    print(f'[SKIP] Final test already exists: {TEST_LOCK}')\n    raise SystemExit(0)"
s=s.replace(old,new)
script.write_text(s)
print('[OK] Runner patched for shared Test-dev and Run-All safety')


In [ ]:

# 30 Optuna trials by default.
os.environ['ACMOT_OPTUNA_TRIALS']='30'
subprocess.run([sys.executable, str(script)], cwd=REPO, check=True)


Results are saved under `MyDrive/AC-MOT-results/optuna_sci_only`. The detector is not trained; only SCI weights are optimized.